# Demand Forecasting & Drift Detection: Full Outputs for Research Paper

This notebook demonstrates the end-to-end outputs of the demand forecasting and inventory pipeline for research reporting.

In [1]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## 2. Data Overview
- Show all processed data files used in the pipeline.
- Display a preview of each file (head, shape, columns).
- Comment on data quality and completeness.

In [2]:
# List of processed data files
processed_files = [
    '../data/processed/daily_demand.csv',
    '../data/processed/forecast_2025.csv',
    '../data/processed/inventory_master.csv',
    '../data/processed/inventory_recommendations.csv',
    '../data/processed/metrics.csv',
    '../data/processed/data_quality.csv'
]

for file in processed_files:
    print(f"\n--- {file} ---")
    try:
        df = pd.read_csv(file)
        display(df.head())
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
    except Exception as e:
        print(f"Could not load {file}: {e}")


--- ../data/processed/daily_demand.csv ---


,Date,SKU,SKU_Name,Demand
0,2024-01-01,APPL-001,Air Fryer 5L,24.0
1,2024-01-01,PETS-001,Dry Dog Food 2kg,8.0
2,2024-01-01,PCAR-003,Shampoo 400ml,11.0
3,2024-01-01,PCAR-002,Razor 5-Blade,7.0
4,2024-01-01,BABY-002,Nappy Pack 40,7.0


Shape: (52632, 4)
Columns: ['Date', 'SKU', 'SKU_Name', 'Demand']

--- ../data/processed/forecast_2025.csv ---


,Date,Forecast_Demand,SKU
0,2025-07-01,12.200358,OUTD-002
1,2025-07-02,13.599988,OUTD-002
2,2025-07-03,12.400271,OUTD-002
3,2025-07-04,11.850358,OUTD-002
4,2025-07-05,11.100506,OUTD-002


Shape: (4392, 3)
Columns: ['Date', 'Forecast_Demand', 'SKU']

--- ../data/processed/inventory_master.csv ---


...

Shape: (2088, 8)
Columns: ['Date', 'SKU', 'Current_Stock', 'In_Transit', 'Recommended_Order_Qty', 'Risk_Level', 'Lead_Time_Days', 'Safety_Stock']

--- ../data/processed/metrics.csv ---


,Date,SKU,Actual,Predicted,MAE,MAPE,RMSE,Drift,Retrained
0,2025-07-03,OUTD-001,11,17.0,6.0,54.5455,6.0,0,0
1,2025-07-03,OFFC-003,13,18.0,5.0,38.4616,5.0,0,0
2,2025-07-03,OFFC-002,13,17.0,4.0,30.7692,4.0,0,0
3,2025-07-03,OFFC-001,14,11.0,3.0,21.4286,3.0,0,0
4,2025-07-03,MUSC-003,12,8.0,4.0,33.3334,4.0,0,0


Shape: (2088, 9)
Columns: ['Date', 'SKU', 'Actual', 'Predicted', 'MAE', 'MAPE', 'RMSE', 'Drift', 'Retrained']

--- ../data/processed/data_quality.csv ---


,sku,date,issues,pass
0,OUTD-001,2025-07-03,[],True
1,OFFC-003,2025-07-03,[],True
2,OFFC-002,2025-07-03,[],True
3,OFFC-001,2025-07-03,[],True
4,MUSC-003,2025-07-03,[],True


Shape: (2088, 4)
Columns: ['sku', 'date', 'issues', 'pass']


## 3. Forecasting Results
- Show the forecast for 2025 (forecast_2025.csv).
- Visualize actual vs. forecasted demand for a few SKUs.
- Summarize forecast accuracy metrics.

In [3]:
{
  "cells": [
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "# Fix: Ensure column names match for merging and plotting\n",
        "# Standardize column names to lowercase for both forecast and actuals\n",
        "df_forecast.columns = [c.lower() for c in df_forecast.columns]\n",
        "df_actual.columns = [c.lower() for c in df_actual.columns]\n",
        "\n",
        "# Pick a few SKUs to visualize\n",
        "sample_skus = df_forecast['sku'].unique()[:3] if 'sku' in df_forecast.columns else []\n",
        "\n",
        "for sku in sample_skus:\n",
        "    plt.figure(figsize=(10,4))\n",
        "    actual = df_actual[df_actual['sku'] == sku]\n",
        "    forecast = df_forecast[df_forecast['sku'] == sku]\n",
        "    if not actual.empty and not forecast.empty:\n",
        "        plt.plot(actual['date'], actual['demand'], label='Actual', marker='o')\n",
        "        plt.plot(forecast['date'], forecast['forecast'], label='Forecast', marker='x')\n",
        "        plt.title(f'SKU: {sku} - Actual vs Forecast')\n",
        "        plt.xlabel('Date')\n",
        "        plt.ylabel('Demand')\n",
        "        plt.legend()\n",
        "        plt.show()\n",
        "    else:\n",
        "        print(f\"No data for SKU {sku} in one of the files.\")\n",
        "\n",
        "# Forecast accuracy metrics (if actuals available for forecast period)\n",
        "if not df_forecast.empty and not df_actual.empty:\n",
        "    merge_cols = ['date', 'sku']\n",
        "    if all(col in df_forecast.columns for col in merge_cols) and all(col in df_actual.columns for col in merge_cols):\n",
        "        merged = pd.merge(df_forecast, df_actual, on=merge_cols, how='inner')\n",
        "        if not merged.empty:\n",
        "            mae = mean_absolute_error(merged['demand'], merged['forecast'])\n",
        "            rmse = mean_squared_error(merged['demand'], merged['forecast'], squared=False)\n",
        "            print(f\"MAE: {mae:.2f}, RMSE: {rmse:.2f}\")\n",
        "        else:\n",
        "            print(\"No overlapping dates between forecast and actuals for metric calculation.\")\n",
        "    else:\n",
        "        print(\"Required columns ['date', 'sku'] not found in both files. Columns in forecast:\", df_forecast.columns.tolist(), \"Columns in actuals:\", df_actual.columns.tolist())"
      ]
    }
  ]
}

{'cells': [{'cell_type': 'code',
   'metadata': {'language': 'python'},
   'source': ['# Fix: Ensure column names match for merging and plotting\n',
    '# Standardize column names to lowercase for both forecast and actuals\n',
    'df_forecast.columns = [c.lower() for c in df_forecast.columns]\n',
    'df_actual.columns = [c.lower() for c in df_actual.columns]\n',
    '\n',
    '# Pick a few SKUs to visualize\n',
    "sample_skus = df_forecast['sku'].unique()[:3] if 'sku' in df_forecast.columns else []\n",
    '\n',
    'for sku in sample_skus:\n',
    '    plt.figure(figsize=(10,4))\n',
    "    actual = df_actual[df_actual['sku'] == sku]\n",
    "    forecast = df_forecast[df_forecast['sku'] == sku]\n",
    '    if not actual.empty and not forecast.empty:\n',
    "        plt.plot(actual['date'], actual['demand'], label='Actual', marker='o')\n",
    "        plt.plot(forecast['date'], forecast['forecast'], label='Forecast', marker='x')\n",
    "        plt.title(f'SKU: {sku} - Actua

## 4. Drift Detection & Data Quality
- Show drift detection results and metrics.
- Visualize drift over time if available.
- Summarize data quality checks.

In [4]:
# Load drift and data quality metrics
try:
    df_metrics = pd.read_csv('../data/processed/metrics.csv')
    display(df_metrics.head())
    if 'drift_score' in df_metrics.columns:
        plt.figure(figsize=(10,4))
        plt.plot(df_metrics['date'], df_metrics['drift_score'], marker='o')
        plt.title('Drift Score Over Time')
        plt.xlabel('Date')
        plt.ylabel('Drift Score')
        plt.show()
except Exception as e:
    print(f"Could not load metrics.csv: {e}")

try:
    df_quality = pd.read_csv('../data/processed/data_quality.csv')
    display(df_quality.head())
except Exception as e:
    print(f"Could not load data_quality.csv: {e}")

,Date,SKU,Actual,Predicted,MAE,MAPE,RMSE,Drift,Retrained
0,2025-07-03,OUTD-001,11,17.0,6.0,54.5455,6.0,0,0
1,2025-07-03,OFFC-003,13,18.0,5.0,38.4616,5.0,0,0
2,2025-07-03,OFFC-002,13,17.0,4.0,30.7692,4.0,0,0
3,2025-07-03,OFFC-001,14,11.0,3.0,21.4286,3.0,0,0
4,2025-07-03,MUSC-003,12,8.0,4.0,33.3334,4.0,0,0


,sku,date,issues,pass
0,OUTD-001,2025-07-03,[],True
1,OFFC-003,2025-07-03,[],True
2,OFFC-002,2025-07-03,[],True
3,OFFC-001,2025-07-03,[],True
4,MUSC-003,2025-07-03,[],True


## 5. Inventory & Recommendations
- Show inventory_master.csv and inventory_recommendations.csv.
- Visualize inventory status distribution.
- Highlight a few example recommendations.

In [5]:
# Load inventory and recommendations
try:
    df_inventory = pd.read_csv('../data/processed/inventory_master.csv')
    display(df_inventory.head())
    if 'status' in df_inventory.columns:
        status_counts = df_inventory['status'].value_counts()
        status_counts.plot(kind='bar', color=['green', 'orange', 'red'])
        plt.title('Inventory Status Distribution')
        plt.xlabel('Status')
        plt.ylabel('Count')
        plt.show()
except Exception as e:
    print(f"Could not load inventory_master.csv: {e}")

try:
    df_recs = pd.read_csv('../data/processed/inventory_recommendations.csv')
    display(df_recs.head(10))
except Exception as e:
    print(f"Could not load inventory_recommendations.csv: {e}")

,SKU,Product,Current_Stock,Lead_Time_Days,Stock_As_Of_Date
0,APPL-001,Air Fryer 5L,330,10,2025-07-01
1,PETS-001,Dry Dog Food 2kg,666,10,2025-07-01
2,PCAR-003,Shampoo 400ml,277,10,2025-07-01
3,PCAR-002,Razor 5-Blade,459,10,2025-07-01
4,BABY-002,Nappy Pack 40,0,7,2025-07-01


,Date,SKU,Current_Stock,In_Transit,Recommended_Order_Qty,Risk_Level,Lead_Time_Days,Safety_Stock
0,2025-07-03,OUTD-001,0,0,129,CRITICAL,7,10
1,2025-07-03,OFFC-003,0,0,136,CRITICAL,7,10
2,2025-07-03,OFFC-002,0,0,128,CRITICAL,7,10
3,2025-07-03,OFFC-001,0,0,86,WARNING,7,10
4,2025-07-03,MUSC-003,0,20,65,WARNING,7,10
5,2025-07-03,MUSC-002,0,0,108,CRITICAL,7,10
6,2025-07-03,HOMK-003,0,0,108,CRITICAL,7,10
7,2025-07-03,MOVI-003,0,0,108,CRITICAL,7,10
8,2025-07-03,MOVI-002,0,0,135,CRITICAL,7,10
9,2025-07-03,MOVI-001,0,0,108,CRITICAL,7,10


## 6. Summary & Discussion
- Summarize key findings from each section.
- Discuss limitations, next steps, and reproducibility.
- Add any additional comments for research reporting.

---

*This notebook provides a full, reproducible overview of the demand forecasting, drift detection, and inventory pipeline. All outputs are generated from the latest pipeline run. For questions or reproducibility, see the README and pipeline scripts.*